# RescueLink AI - Audio Pipeline Testing & Operations
This notebook covers manual operational testing for the local quantized Whisper + emergency classification microservice:
1. **Runtime Preflight** - Check provider mode, ffmpeg, and local environment readiness
2. **Audio File Validation** - Enforce constraints before API calls
3. **Fallback Behavior** - Validate graceful degradation and clear errors
4. **API Manual Test Flow** - `/health` → `/classify` → `/v1/transcribe` → `/v1/classify-audio` → `/v1/audio/stats`

**Status**: Local-first quantized Whisper mode (internet-independent when API fallback is disabled).

In [ ]:
# Standard library
import os
import sys
import json
import logging
import time
import shutil
from pathlib import Path
from datetime import datetime

# Data & audio
import numpy as np
try:
    import librosa
except ModuleNotFoundError:
    librosa = None
try:
    import soundfile as sf
except ModuleNotFoundError:
    sf = None

# HTTP / diagnostics
import requests

# Visualization
import matplotlib.pyplot as plt
import pandas as pd

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Add parent directory to path
sys.path.insert(0, str(Path().resolve().parent))

print("✓ Imports complete")
print(f"  - NumPy: {np.__version__}")
print(f"  - Librosa available: {librosa is not None}")
print(f"  - SoundFile available: {sf is not None}")

# Runtime preflight (safe to print: no secret values)
api_base = os.getenv("AI_TEST_BASE_URL", "http://localhost:8000")
stt_provider = os.getenv("STT_PROVIDER", "local")
stt_fallback = os.getenv("STT_ENABLE_API_FALLBACK", "false")
stt_device = os.getenv("STT_DEVICE", "auto")
stt_compute = os.getenv("STT_COMPUTE_TYPE", "auto")
stt_model_size = os.getenv("STT_LOCAL_MODEL_SIZE", "medium")
ffmpeg_ok = shutil.which("ffmpeg") is not None

print("\n✓ Runtime preflight")
print(f"  - API base: {api_base}")
print(f"  - STT provider: {stt_provider}")
print(f"  - API fallback enabled: {stt_fallback}")
print(f"  - STT device: {stt_device}")
print(f"  - STT compute type: {stt_compute}")
print(f"  - STT model size: {stt_model_size}")
print(f"  - ffmpeg on PATH: {ffmpeg_ok}")

if librosa is None or sf is None:
    print("\n⚠ Optional notebook packages missing. Install in notebook if needed:")
    print("  %pip install librosa soundfile")

ModuleNotFoundError: No module named 'librosa'

## Section 1: Setup Monitoring and Logging

Configure logging and monitoring infrastructure for the audio pipeline:
- Track API call latency
- Monitor prediction confidence scores
- Log processing times for each stage
- Set up alerts for low-confidence predictions

In [ ]:
class MonitoringTracker:
    """Track API usage, latency, and prediction confidence"""
    
    def __init__(self):
        self.calls = []
        self.start_time = time.time()
    
    def log_call(self, call_type: str, duration: float, confidence: float = None, 
                 status: str = "success", error: str = None):
        """Log an API call"""
        record = {
            "timestamp": datetime.now().isoformat(),
            "call_type": call_type,
            "duration_seconds": duration,
            "confidence": confidence,
            "status": status,
            "error": error,
        }
        self.calls.append(record)
        
        if status == "success" and confidence is not None and confidence < 0.7:
            logger.warning(f"⚠ Low confidence prediction: {confidence:.2f}")
        elif status == "error":
            logger.error(f"✗ {call_type} failed: {error}")
        else:
            logger.info(f"✓ {call_type} | latency: {duration:.2f}s | confidence: {confidence:.2f}")
    
    def get_summary(self):
        """Get monitoring summary"""
        if not self.calls:
            return None
        
        df = pd.DataFrame(self.calls)
        return {
            "total_calls": len(df),
            "success_rate": (df["status"] == "success").sum() / len(df) * 100,
            "avg_latency": df[df["status"] == "success"]["duration_seconds"].mean(),
            "avg_confidence": df[df["status"] == "success"]["confidence"].mean(),
            "low_confidence_calls": (df["confidence"] < 0.7).sum(),
            "failed_calls": (df["status"] == "error").sum(),
        }
    
    def plot_metrics(self):
        """Plot monitoring metrics"""
        if not self.calls:
            print("No data to plot")
            return
        
        df = pd.DataFrame(self.calls)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Latency over time
        success_df = df[df["status"] == "success"]
        axes[0, 0].plot(success_df["duration_seconds"])
        axes[0, 0].set_title("API Latency Over Time")
        axes[0, 0].set_ylabel("Latency (seconds)")
        axes[0, 0].grid(True, alpha=0.3)
        
        # Confidence distribution
        axes[0, 1].hist(success_df["confidence"].dropna(), bins=20, edgecolor='black')
        axes[0, 1].set_title("Confidence Score Distribution")
        axes[0, 1].set_xlabel("Confidence")
        axes[0, 1].axvline(x=0.7, color='r', linestyle='--', label='Low threshold')
        axes[0, 1].legend()
        
        # Call status breakdown
        status_counts = df["status"].value_counts()
        axes[1, 0].bar(status_counts.index, status_counts.values, color=['green', 'red'])
        axes[1, 0].set_title("Call Status Distribution")
        axes[1, 0].set_ylabel("Count")
        
        # Latency vs Confidence scatter
        axes[1, 1].scatter(success_df["duration_seconds"], success_df["confidence"], alpha=0.6)
        axes[1, 1].set_title("Latency vs Confidence")
        axes[1, 1].set_xlabel("Latency (seconds)")
        axes[1, 1].set_ylabel("Confidence")
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

# Initialize monitoring
monitor = MonitoringTracker()
print("✓ Monitoring tracker initialized")

## Section 2: Implement Audio File Validation

Enforce strict constraints on audio files:
- **Duration**: 15-60 seconds (fast iteration for testing)
- **File Size**: Maximum 25MB (API limits)
- **Format**: .wav, .mp3, .m4a, .flac, .ogg supported
- **Sample Rate**: Auto-resampled by librosa

In [ ]:
class AudioValidator:
    """Validate audio files before processing"""
    
    SUPPORTED_FORMATS = {'.wav', '.mp3', '.m4a', '.flac', '.ogg'}
    MIN_DURATION = 15  # seconds (reduced for faster testing)
    MAX_DURATION = 60  # seconds
    MAX_FILE_SIZE_MB = 25
    
    @classmethod
    def validate_file_path(cls, file_path: str) -> dict:
        """Validate audio file"""
        file_path = Path(file_path)
        errors = []
        warnings = []
        
        # Check existence
        if not file_path.exists():
            errors.append(f"File not found: {file_path}")
            return {"valid": False, "errors": errors, "warnings": warnings}
        
        # Check format
        if file_path.suffix.lower() not in cls.SUPPORTED_FORMATS:
            errors.append(f"Unsupported format: {file_path.suffix}. "
                        f"Supported: {', '.join(cls.SUPPORTED_FORMATS)}")
        
        # Check file size
        file_size_mb = file_path.stat().st_size / (1024 * 1024)
        if file_size_mb > cls.MAX_FILE_SIZE_MB:
            errors.append(f"File too large: {file_size_mb:.1f}MB (max: {cls.MAX_FILE_SIZE_MB}MB)")
        
        # Load and check duration
        try:
            y, sr = librosa.load(str(file_path), sr=None)
            duration = librosa.get_duration(y=y, sr=sr)
            
            if duration < cls.MIN_DURATION:
                errors.append(f"Audio too short: {duration:.1f}s (min: {cls.MIN_DURATION}s)")
            if duration > cls.MAX_DURATION:
                errors.append(f"Audio too long: {duration:.1f}s (max: {cls.MAX_DURATION}s)")
            
            # Check for silence
            if len(y) == 0:
                errors.append("Audio file is empty or corrupted")
            
            # Warnings for edge cases
            if duration < 20:
                warnings.append(f"Short audio ({duration:.1f}s). Transcription may be incomplete.")
            if duration > 55:
                warnings.append(f"Long audio ({duration:.1f}s). Processing may take longer.")
            
        except Exception as e:
            errors.append(f"Failed to load audio: {e}")
        
        return {
            "valid": len(errors) == 0,
            "file_size_mb": file_size_mb,
            "duration": duration if 'duration' in locals() else None,
            "errors": errors,
            "warnings": warnings,
        }

# Test validator
print("=" * 60)
print("AUDIO VALIDATION EXAMPLE")
print("=" * 60)

# Create a test audio file
test_audio_path = "test_audio_45sec.wav"
duration_sec = 45
sr = 16000
t = np.linspace(0, duration_sec, sr * duration_sec)
# Mix of low-frequency tone and speech-like noise
tone = 0.3 * np.sin(2 * np.pi * 100 * t)  # 100 Hz tone
noise = 0.2 * np.random.normal(0, 1, len(t))  # Gaussian noise
audio = tone + noise

sf.write(test_audio_path, audio, sr)
print(f"Test audio created: {test_audio_path} ({duration_sec}s)")

# Validate
result = AudioValidator.validate_file_path(test_audio_path)
print(f"\nValidation result: {result['valid']}")
print(f"File size: {result['file_size_mb']:.2f}MB")
print(f"Duration: {result['duration']:.1f}s")
if result['warnings']:
    for w in result['warnings']:
        print(f"  ⚠ {w}")
if result['errors']:
    for e in result['errors']:
        print(f"  ✗ {e}")

print("=" * 60)

## Section 3: Add Fallback Mechanisms

Implement graceful degradation:
- If Whisper transcription fails → Return 503 error (service unavailable)
- User can then use text-only `/classify` endpoint
- Log all fallback events for monitoring
- Provide clear error messages to API consumers

In [ ]:
class FallbackHandler:
    """Handle service failures gracefully"""
    
    @staticmethod
    def handle_transcription_failure(error: str, audio_duration: float) -> dict:
        """Handle transcription failure with fallback instructions"""
        return {
            "status": "transcription_failed",
            "http_status": 503,
            "error_message": f"Speech-to-text service unavailable: {error}",
            "fallback_action": "Use text-only endpoint (/classify) with manual transcription",
            "details": {
                "audio_duration": audio_duration,
                "timestamp": datetime.now().isoformat(),
                "recommendation": "User can manually transcribe audio and use text endpoint"
            }
        }
    
    @staticmethod
    def handle_classification_failure(error: str) -> dict:
        """Handle classification failure with retry instructions"""
        return {
            "status": "classification_failed",
            "http_status": 500,
            "error_message": f"Classification error: {error}",
            "fallback_action": "Retry request or use different audio",
            "details": {
                "timestamp": datetime.now().isoformat(),
                "recommendation": "Ensure audio is clear and contains emergency-related content"
            }
        }
    
    @staticmethod
    def handle_validation_failure(validation_result: dict) -> dict:
        """Handle validation failure with specific feedback"""
        errors = "\n  ".join(validation_result.get("errors", []))
        return {
            "status": "validation_failed",
            "http_status": 400,
            "error_message": f"Audio validation failed:\n  {errors}",
            "requirements": {
                "duration_seconds": f"{AudioValidator.MIN_DURATION}-{AudioValidator.MAX_DURATION}",
                "max_file_size_mb": AudioValidator.MAX_FILE_SIZE_MB,
                "supported_formats": list(AudioValidator.SUPPORTED_FORMATS)
            },
            "details": validation_result
        }

# Demo fallback scenarios
print("=" * 60)
print("FALLBACK HANDLING EXAMPLES")
print("=" * 60)

# Scenario 1: Transcription failure
print("\n1. Transcription Service Unavailable:")
fallback1 = FallbackHandler.handle_transcription_failure(
    error="HF API timeout after 60s",
    audio_duration=45.2
)
print(json.dumps(fallback1, indent=2))

# Scenario 2: Classification failure
print("\n2. Classification Error:")
fallback2 = FallbackHandler.handle_classification_failure(
    error="Model inference timeout"
)
print(json.dumps(fallback2, indent=2))

# Scenario 3: Validation failure
print("\n3. Validation Failure:")
validation_result = {
    "valid": False,
    "errors": ["Audio too long: 75.5s (max: 60s)", "File too large: 30.2MB (max: 25MB)"]
}
fallback3 = FallbackHandler.handle_validation_failure(validation_result)
print(json.dumps(fallback3, indent=2))

print("=" * 60)

## Section 4: Test Edge Cases and Error Handling

Test the system with various challenging scenarios:
- **Corrupted audio files** - Invalid format or truncated data
- **Noisy audio** - Low SNR, background noise
- **Edge case durations** - Too short (<30s), too long (>60s)
- **Empty/Silent audio** - No speech content
- **Mixed languages** - Filipino + English code-switching

In [ ]:
class EdgeCaseTester:
    """Test edge cases and error handling"""
    
    @staticmethod
    def create_test_audio(scenario: str, duration: int = 30) -> str:
        """Create test audio for different scenarios"""
        sr = 16000
        t = np.linspace(0, duration, sr * duration)
        
        if scenario == "clean_speech":
            # Simulate clean speech (fundamental ~100-200 Hz)
            audio = 0.5 * np.sin(2 * np.pi * 150 * t) * np.exp(-t / 10)
        
        elif scenario == "noisy":
            # High noise
            tone = 0.3 * np.sin(2 * np.pi * 150 * t)
            noise = 0.5 * np.random.normal(0, 1, len(t))  # 50% noise
            audio = tone + noise
        
        elif scenario == "silent":
            # Silent or near-silent
            audio = 0.01 * np.random.normal(0, 1, len(t))
        
        elif scenario == "mixed_language":
            # Simulate alternating English/Filipino-like patterns
            first_half = 0.3 * np.sin(2 * np.pi * 120 * t[:len(t)//2])
            second_half = 0.3 * np.sin(2 * np.pi * 180 * t[len(t)//2:])
            audio = np.concatenate([first_half, second_half])
        
        else:
            audio = np.zeros(len(t))
        
        # Normalize
        audio = audio / (np.max(np.abs(audio)) + 1e-8) * 0.8
        
        filename = f"test_{scenario}_{duration}s.wav"
        sf.write(filename, audio, sr)
        return filename

# Run edge case tests
print("=" * 60)
print("EDGE CASE TESTING")
print("=" * 60)

test_cases = [
    ("clean_speech", 45, "Normal emergency report (clean audio)"),
    ("noisy", 45, "Noisy environment (high background noise)"),
    ("silent", 40, "Silent/empty audio (no speech)"),
    ("mixed_language", 50, "Mixed Filipino+English"),
]

test_results = []

for scenario, duration, description in test_cases:
    print(f"\nTest: {description}")
    print(f"  Creating {duration}s audio ({scenario})...")
    
    audio_path = EdgeCaseTester.create_test_audio(scenario, duration)
    validation = AudioValidator.validate_file_path(audio_path)
    
    print(f"  File size: {validation['file_size_mb']:.2f}MB")
    print(f"  Duration: {validation['duration']:.1f}s")
    print(f"  Valid: {validation['valid']}")
    
    if validation['errors']:
        print(f"  Errors:")
        for e in validation['errors']:
            print(f"    ✗ {e}")
    
    if validation['warnings']:
        print(f"  Warnings:")
        for w in validation['warnings']:
            print(f"    ⚠ {w}")
    
    test_results.append({
        "scenario": scenario,
        "valid": validation['valid'],
        "duration": validation['duration'],
        "errors": len(validation['errors']),
        "warnings": len(validation['warnings'])
    })

# Summary
print("\n" + "=" * 60)
print("TEST SUMMARY")
print("=" * 60)
results_df = pd.DataFrame(test_results)
print(results_df.to_string(index=False))
print(f"\nPassed: {results_df['valid'].sum()}/{len(results_df)}")

# Cleanup
import glob
for f in glob.glob("test_*.wav"):
    os.remove(f)
print("✓ Test files cleaned up")
print("=" * 60)

## Summary & Manual Validation Checklist

**Current Readiness:**

- ✅ Local-first Whisper provider configuration checks
- ✅ Audio validation helpers for local testing
- ✅ Fallback/error handling simulation
- ✅ Endpoint manual flow aligned to current API contracts
- ✅ Stats endpoint verification (`/v1/audio/stats`)

**Manual Test Flow (recommended):**
1. Ensure server is running: `python -m uvicorn api.main:app --reload --port 8000`
2. Verify local mode config (`STT_PROVIDER=local`, `STT_ENABLE_API_FALLBACK=false` for offline-first)
3. Run endpoint sequence cells: `/health`, `/classify`, `/v1/transcribe`, `/v1/classify-audio`, `/v1/audio/stats`
4. Review latency/confidence outputs and warnings
5. Capture findings for backlog items AI-T603 / AI-T709 manual validation evidence

## Section 5: Manual API Flow (Local Quantized Whisper)

This section validates the live API path used in production-compatible tests:
- `GET /health`
- `POST /classify` (text baseline)
- `POST /v1/transcribe` (file upload)
- `POST /v1/classify-audio` (file upload)
- `GET /v1/audio/stats`

**Requirements:**
- API running: `python -m uvicorn api.main:app --reload --port 8000`
- Local STT mode configured (`STT_PROVIDER=local`)
- Optional strict offline: `STT_ENABLE_API_FALLBACK=false`
- Test audio file available (this notebook auto-discovers sample files)

**Note:** This replaces the old `/v1/transcribe-mic` path (not part of current API).

In [ ]:
# Manual API test flow: health -> classify -> transcribe -> classify-audio -> stats
import requests
from pathlib import Path
import time

API_BASE = os.getenv("AI_TEST_BASE_URL", "http://localhost:8000")
AI_TOKEN = os.getenv("AI_INTERNAL_TOKEN", "").strip()

def build_headers(json_mode=False):
    headers = {}
    if AI_TOKEN:
        headers["x-ai-service-token"] = AI_TOKEN
    if json_mode:
        headers["Content-Type"] = "application/json"
    return headers

def print_step(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

def pick_audio_file():
    candidates = []
    search_roots = [Path('.'), Path('test'), Path('../Backend/uploads/incidents')]
    patterns = ["*.wav", "*.m4a", "*.mp3", "*.flac", "*.ogg"]
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            candidates.extend(sorted(root.glob(pattern)))
    # Prefer known test fixtures if present
    for preferred in [
        Path("test/test_report_1.m4a"),
        Path("test/test_report_2.m4a"),
        Path("test_audio_45sec.wav"),
    ]:
        if preferred.exists():
            return preferred
    return candidates[0] if candidates else None

# 1) Health
print_step("1) GET /health")
t0 = time.time()
health_resp = requests.get(f"{API_BASE}/health", headers=build_headers(), timeout=30)
print("Status:", health_resp.status_code, "| Latency:", f"{time.time()-t0:.2f}s")
try:
    print(json.dumps(health_resp.json(), indent=2))
except Exception:
    print(health_resp.text)

# 2) Text classification baseline
print_step("2) POST /classify")
sample_text = "May sunog sa barangay at may nasugatan, kailangan ng tulong agad."
payload = {"text": sample_text, "threshold": 0.3}
t0 = time.time()
classify_resp = requests.post(
    f"{API_BASE}/classify",
    headers=build_headers(json_mode=True),
    json=payload,
    timeout=60,
    )
print("Status:", classify_resp.status_code, "| Latency:", f"{time.time()-t0:.2f}s")
try:
    classify_json = classify_resp.json()
    print(json.dumps(classify_json, indent=2)[:2000])
except Exception:
    classify_json = {}
    print(classify_resp.text)

# 3) Transcribe audio file
print_step("3) POST /v1/transcribe")
audio_path = pick_audio_file()
if audio_path is None:
    raise FileNotFoundError("No test audio found. Add one under RescueLink AI/test or current folder.")
print("Using audio:", str(audio_path))

with open(audio_path, "rb") as fh:
    files = {"file": (audio_path.name, fh, "application/octet-stream")}
    t0 = time.time()
    transcribe_resp = requests.post(
        f"{API_BASE}/v1/transcribe",
        headers=build_headers(),
        files=files,
        timeout=180,
    )
print("Status:", transcribe_resp.status_code, "| Latency:", f"{time.time()-t0:.2f}s")
try:
    transcribe_json = transcribe_resp.json()
    print(json.dumps(transcribe_json, indent=2)[:2000])
except Exception:
    transcribe_json = {}
    print(transcribe_resp.text)

CLASSIFY_TEXT = transcribe_json.get("transcription", "")
print("\nCLASSIFY_TEXT prepared:", bool(CLASSIFY_TEXT))

# 4) Full classify-audio path
print_step("4) POST /v1/classify-audio")
with open(audio_path, "rb") as fh:
    files = {"file": (audio_path.name, fh, "application/octet-stream")}
    t0 = time.time()
    classify_audio_resp = requests.post(
        f"{API_BASE}/v1/classify-audio",
        headers=build_headers(),
        files=files,
        timeout=240,
    )
print("Status:", classify_audio_resp.status_code, "| Latency:", f"{time.time()-t0:.2f}s")
try:
    classify_audio_json = classify_audio_resp.json()
    print(json.dumps(classify_audio_json, indent=2)[:2200])
except Exception:
    classify_audio_json = {}
    print(classify_audio_resp.text)

# 5) Audio stats endpoint
print_step("5) GET /v1/audio/stats")
t0 = time.time()
stats_resp = requests.get(f"{API_BASE}/v1/audio/stats", headers=build_headers(), timeout=30)
print("Status:", stats_resp.status_code, "| Latency:", f"{time.time()-t0:.2f}s")
try:
    print(json.dumps(stats_resp.json(), indent=2))
except Exception:
    print(stats_resp.text)

print("\n✓ Manual API flow complete")

🎤 MICROPHONE RECORDING
Duration: 15 seconds
Sample Rate: 16000 Hz

🔴 RECORDING NOW - Speak clearly into your microphone!

✓ Recording complete (23.4s total)
Status: 200

✓ TRANSCRIPTION SUCCESSFUL
------------------------------------------------------------
 May tao po na nag-aaway dito sa kalsada. Road to Rage. Dito po sa kanto ng Jollibee Junction. Road to Rage. May nag-aaway po dito.
------------------------------------------------------------
Duration: 15.0s
Latency: 5.9s
Confidence: 1.00
Language: auto

CLASSIFY_TEXT ready for classification
Saved WAV: C:\Users\Aaron\AppData\Local\Temp\tmpe1ige5io.wav


In [ ]:
# Manual classification replay using transcribed CLASSIFY_TEXT
import requests
import json

API_BASE = os.getenv("AI_TEST_BASE_URL", "http://localhost:8000")
AI_TOKEN = os.getenv("AI_INTERNAL_TOKEN", "").strip()

def headers_for_json():
    headers = {"Content-Type": "application/json"}
    if AI_TOKEN:
        headers["x-ai-service-token"] = AI_TOKEN
    return headers

if not CLASSIFY_TEXT or not str(CLASSIFY_TEXT).strip():
    print("No transcription available. Run the previous cell first.")
else:
    payload = {"text": CLASSIFY_TEXT, "threshold": 0.3}
    print("=" * 70)
    print("EMERGENCY CLASSIFICATION REPLAY")
    print("=" * 70)
    print("\nMessage for Human Verification:")
    print("-" * 70)
    print(CLASSIFY_TEXT)
    print("-" * 70)

    resp = requests.post(f"{API_BASE}/classify", json=payload, headers=headers_for_json(), timeout=60)

    print("\n" + "=" * 70)
    print("CLASSIFICATION RESULTS")
    print("=" * 70)
    print(f"Status: {resp.status_code}")

    if resp.status_code == 200:
        result = resp.json()
        print(f"\n📋 Original Message: {result.get('message', 'N/A')}")
        print(f"\n🚨 Incident Types: {', '.join(result.get('incident_types', []))}")
        print(f"⚠️  Severity: {result.get('severity_color', 'Unknown')}")
        print(f"\n🤖 Model: {result.get('model_version', 'Unknown')}")
        print("\n📊 Confidence Scores:")
        for incident_type, score in result.get('confidence_scores', {}).items():
            bar = "█" * int(float(score) * 20)
            print(f"  {incident_type:.<20} {float(score):.2%} {bar}")
    else:
        try:
            print(json.dumps(resp.json(), indent=2))
        except Exception:
            print(resp.text)

    print("=" * 70)

EMERGENCY CLASSIFICATION REQUEST

Message for Human Verification:
----------------------------------------------------------------------
 May tao po na nag-aaway dito sa kalsada. Road to Rage. Dito po sa kanto ng Jollibee Junction. Road to Rage. May nag-aaway po dito.
----------------------------------------------------------------------

Sending classification request...

CLASSIFICATION RESULTS
Status: 200

📋 Original Message:  May tao po na nag-aaway dito sa kalsada. Road to Rage. Dito po sa kanto ng Jollibee Junction. Road to Rage. May nag-aaway po dito.

🚨 Incident Types: Other
⚠️  Severity: 🔴 Immediate

🤖 Model: 2.0.0-xlm-roberta-filipino

📊 Confidence Scores:
  Fire................ 1.29% 
  Crime............... 48.89% █████████
  Accident............ 4.21% 
  Medical............. 0.84% 
  Natural Disaster.... 1.78% 
  Other............... 99.14% ███████████████████
